# Topology Adaptive GCN (TAGCN) on Cora

Classify papers in the Cora citation network by topic. TAGCN
([Du et al., 2017](https://arxiv.org/abs/1710.10370)) learns a separate filter for each neighborhood
distance (1 to 3 hops) instead of relying only on direct neighbors.

Same model as PyG's [`examples/tagcn.py`](https://github.com/pyg-team/pytorch_geometric/blob/master/examples/tagcn.py).

In [ ]:
# Install K3-Node, then choose a backend: "tensorflow", "torch" or "jax"
!pip install -q "k3-node[examples] @ git+https://github.com/anas-rz/k3-node"
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

## Load the data

`data` holds one graph: node features `x`, edges `edge_index`, labels `y`, and masks marking the training, validation and test nodes.

In [ ]:
import keras
from keras import ops
from k3_node.datasets import Planetoid
from k3_node.loader import FullGraphDataset
from k3_node.transforms import NormalizeFeatures

dataset = Planetoid("data/Planetoid", name="Cora", transform=NormalizeFeatures())
data = dataset[0]
print(data)

## Define the model

In [ ]:
from k3_node.layers import TAGConv


class TAGCN(keras.Model):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = TAGConv(in_channels, 16)
        self.conv2 = TAGConv(16, out_channels)
        self.dropout = keras.layers.Dropout(0.5)

    def call(self, data, training=False):
        x = ops.relu(self.conv1(data.x, data.edge_index))
        x = self.dropout(x, training=training)
        return self.conv2(x, data.edge_index)


model = TAGCN(dataset.num_features, dataset.num_classes)

## Train

`FullGraphDataset` feeds the whole graph to Keras; `mask` selects which nodes count in the loss and the accuracy.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01, weight_decay=0.0005),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    weighted_metrics=["accuracy"],
)
model.fit(
    FullGraphDataset(data, mask="train_mask"),
    validation_data=FullGraphDataset(data, mask="val_mask"),
    epochs=200,
    verbose=2,
)

## Evaluate

In [ ]:
loss, accuracy = model.evaluate(FullGraphDataset(data, mask="test_mask"), verbose=0)
print(f"Test accuracy: {accuracy:.4f}")